# MedoraAI — Multi-Task CXR Training
### CheXpert (14-class classification) + SIIM-ACR/CANDID-PTX (pneumothorax segmentation)

See `TRAINING_GUIDE.md` for the full explanation. This notebook:
1. Sets up environment + config
2. Downloads/points to datasets (SIIM-ACR via Kaggle API scripted; CheXpert/CANDID require manual auth download)
3. Builds Dataset/DataLoader classes
4. Builds the multi-task model (shared backbone + classification head + segmentation head)
5. Trains with per-epoch checkpointing
6. Runs a final smoke test on the saved `.pt` file

> **Before running:** fill in the `CONFIG` cell paths. Datasets requiring authentication
> (CheXpert, MIMIC-CXR, CANDID-PTX) must be downloaded manually first per their data use
> agreements — this notebook cannot bypass that.

In [ ]:
# ---- 0. Install dependencies ----
!pip install -q torch torchvision torchmetrics pandas numpy pillow scikit-learn matplotlib opencv-python albumentations kaggle tqdm

In [ ]:
import os, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
# ---- 1. CONFIG — edit these paths before running ----

CONFIG = {
    # --- CheXpert (manual download required: https://stanfordaimi.azurewebsites.net/datasets/8cbd9ed4-2eb9-4565-affc-111cf4f7ebe2) ---
    "CHEXPERT_ROOT": "/data/chexpert",              # expects train.csv/valid.csv + image folders inside
    "CHEXPERT_TRAIN_CSV": "/data/chexpert/train.csv",
    "CHEXPERT_VALID_CSV": "/data/chexpert/valid.csv",

    # --- SIIM-ACR Pneumothorax Segmentation (Kaggle, scripted download below) ---
    "SIIM_ROOT": "/data/siim_acr",

    # --- CANDID-PTX (manual download required: https://doi.org/10.7488/ds/3038) ---
    "CANDID_ROOT": "/data/candid_ptx",

    # --- training params ---
    "IMG_SIZE": 320,
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 8,
    "LR": 1e-4,
    "WEIGHT_DECAY": 1e-5,
    "SEG_LOSS_WEIGHT": 1.0,       # lambda in cls_loss + lambda * seg_loss
    "PHASE_A_EPOCHS": 10,          # CheXpert-only base classifier
    "PHASE_B_EPOCHS": 15,          # multi-task fine-tune with seg head
    "CHECKPOINT_DIR": "./checkpoints",
    "LOG_DIR": "./logs",
    "EARLY_STOP_PATIENCE": 3,
}

os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)
os.makedirs(CONFIG["LOG_DIR"], exist_ok=True)

CHEXPERT_CLASSES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"
]
PNEUMOTHORAX_IDX = CHEXPERT_CLASSES.index("Pneumothorax")
print(f"Pneumothorax is class index {PNEUMOTHORAX_IDX} of {len(CHEXPERT_CLASSES)}")

## 2. Dataset download

- **CheXpert / CANDID-PTX**: requires manual registration + download (data use agreements).
  Download the zip(s), extract to the paths set in `CONFIG` above, then skip the cells below.
- **SIIM-ACR**: can be pulled via the Kaggle API if you have a `kaggle.json` token placed at
  `~/.kaggle/kaggle.json`.

In [ ]:
# ---- 2a. SIIM-ACR download via Kaggle API ----
# Requires ~/.kaggle/kaggle.json with your Kaggle API credentials.
# Get one at: https://www.kaggle.com/settings -> API -> Create New Token

KAGGLE_JSON = Path.home() / ".kaggle" / "kaggle.json"
if not KAGGLE_JSON.exists():
    print("!! No ~/.kaggle/kaggle.json found. Upload your Kaggle API token there, then re-run this cell.")
    print("   Skipping SIIM-ACR auto-download for now.")
else:
    os.chmod(KAGGLE_JSON, 0o600)
    siim_dir = Path(CONFIG["SIIM_ROOT"])
    siim_dir.mkdir(parents=True, exist_ok=True)
    if not any(siim_dir.iterdir()):
        !kaggle competitions download -c siim-acr-pneumothorax-segmentation -p {siim_dir}
        !cd {siim_dir} && unzip -q -o '*.zip' && rm -f *.zip
        print("SIIM-ACR downloaded and extracted to", siim_dir)
    else:
        print("SIIM-ACR directory already populated, skipping download.")

In [ ]:
# ---- 2b. Verify manual-download datasets are present ----
def check_path(path, name):
    p = Path(path)
    ok = p.exists() and any(p.iterdir()) if p.exists() else False
    print(f"[{'OK' if ok else 'MISSING'}] {name}: {path}")
    return ok

chexpert_ok = check_path(CONFIG["CHEXPERT_ROOT"], "CheXpert")
candid_ok = check_path(CONFIG["CANDID_ROOT"], "CANDID-PTX")
siim_ok = check_path(CONFIG["SIIM_ROOT"], "SIIM-ACR")

if not chexpert_ok:
    print("\n  -> Register + download CheXpert manually: https://stanfordaimi.azurewebsites.net/datasets/8cbd9ed4-2eb9-4565-affc-111cf4f7ebe2")
if not candid_ok:
    print("  -> Register + download CANDID-PTX manually: https://doi.org/10.7488/ds/3038")

## 3. Dataset classes

In [ ]:
IMG_SIZE = CONFIG["IMG_SIZE"]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.0),  # NOTE: do NOT flip CXRs by default — breaks laterality (L/R matters clinically)
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class CheXpertDataset(Dataset):
    """Classification-only dataset. Expects a CSV with an image Path column
    and one column per class in CHEXPERT_CLASSES (values in {0,1,-1,nan});
    -1 (uncertain) is mapped to 1 (U-Ones policy, a common CheXpert baseline)."""
    def __init__(self, csv_path, root, classes, transform):
        self.df = pd.read_csv(csv_path)
        self.root = Path(root)
        self.classes = classes
        self.transform = transform
        for c in classes:
            if c in self.df.columns:
                self.df[c] = self.df[c].fillna(0).replace(-1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root / row["Path"]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        labels = torch.tensor([row.get(c, 0) for c in self.classes], dtype=torch.float32)
        return {"image": img, "labels": labels, "has_mask": torch.tensor(0.0), "mask": torch.zeros(1, IMG_SIZE, IMG_SIZE)}


class PneumothoraxSegDataset(Dataset):
    """SIIM-ACR / CANDID-PTX style dataset with pixel masks.
    Expects a CSV with columns: image_path, mask_path (mask_path can be empty for negatives ->
    all-zero mask), and a binary pneumothorax label."""
    def __init__(self, csv_path, img_root, mask_root, classes, transform, img_size):
        self.df = pd.read_csv(csv_path)
        self.img_root = Path(img_root)
        self.mask_root = Path(mask_root)
        self.classes = classes
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_root / row["image_path"]).convert("RGB")
        img = self.transform(img)

        if isinstance(row.get("mask_path", None), str) and len(row["mask_path"]) > 0:
            mask = cv2.imread(str(self.mask_root / row["mask_path"]), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)

        labels = torch.zeros(len(self.classes), dtype=torch.float32)
        labels[PNEUMOTHORAX_IDX] = float(row.get("pneumothorax", int(mask.sum() > 0)))

        return {"image": img, "labels": labels, "has_mask": torch.tensor(1.0), "mask": mask}


print("Dataset classes ready. Build CSV manifests (image_path, mask_path, pneumothorax) for SIIM/CANDID before training —")
print("SIIM-ACR ships RLE-encoded masks in train-rle.csv; decode these to PNG masks first (see helper below).")

In [ ]:
# ---- Helper: decode SIIM-ACR RLE masks to PNGs (run once if using raw SIIM-ACR export) ----
def rle2mask(rle, width, height):
    mask = np.zeros(width * height, dtype=np.uint8)
    if rle.strip() == "-1":
        return mask.reshape(height, width)
    s = rle.split()
    starts, lengths = [int(x) for x in s[0::2]], [int(x) for x in s[1::2]]
    for start, length in zip(starts, lengths):
        mask[start:start + length] = 255
    return mask.reshape(width, height).T

def build_siim_manifest(siim_root, out_csv):
    """Run once to convert SIIM-ACR's train-rle.csv into an (image_path, mask_path, pneumothorax) manifest.
    Adjust glob patterns if your SIIM-ACR export layout differs."""
    rle_csv = Path(siim_root) / "train-rle.csv"
    if not rle_csv.exists():
        print("train-rle.csv not found — skipping manifest build (check SIIM_ROOT layout).")
        return
    rle_df = pd.read_csv(rle_csv)
    rle_df.columns = [c.strip() for c in rle_df.columns]
    mask_dir = Path(siim_root) / "masks_png"
    mask_dir.mkdir(exist_ok=True)
    rows = []
    for _, r in tqdm(rle_df.iterrows(), total=len(rle_df), desc="Decoding RLE masks"):
        img_id = r["ImageId"]
        rle = str(r[" EncodedPixels"]) if " EncodedPixels" in r else str(r["EncodedPixels"])
        has_ptx = rle.strip() != "-1"
        mask_path = f"{img_id}.png"
        if has_ptx:
            m = rle2mask(rle, 1024, 1024)
            cv2.imwrite(str(mask_dir / mask_path), m)
        rows.append({
            "image_path": f"images/{img_id}.png",
            "mask_path": f"masks_png/{mask_path}" if has_ptx else "",
            "pneumothorax": int(has_ptx),
        })
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"Manifest written to {out_csv} ({len(rows)} rows)")

# Example (uncomment once SIIM-ACR is downloaded and images/ folder is flattened to PNGs):
# build_siim_manifest(CONFIG["SIIM_ROOT"], f"{CONFIG['SIIM_ROOT']}/manifest.csv")

## 4. Model — shared backbone + classification head + segmentation head

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class MultiTaskCXRModel(nn.Module):
    """DenseNet-121 backbone shared between a 14-class classification head
    and a lightweight decoder segmentation head (pneumothorax only)."""
    def __init__(self, num_classes=14, pretrained=True):
        super().__init__()
        backbone = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = backbone.features            # output: [B, 1024, H/32, W/32]
        self.feat_channels = 1024

        # classification head
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(self.feat_channels, num_classes)
        )

        # segmentation decoder (simple upsampling decoder, not full U-Net — fine for
        # aux supervision purposes; upgrade to U-Net decoder w/ skip connections if
        # segmentation quality needs to improve further)
        self.seg_decoder = nn.Sequential(
            ConvBlock(self.feat_channels, 512), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(512, 256), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(256, 128), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(128, 64), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(64, 32), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x):
        feat = self.features(x)                 # [B, 1024, H/32, W/32]
        feat_act = F.relu(feat, inplace=False)   # DenseNet convention before pooling
        cls_logits = self.cls_head(feat_act)
        seg_logits = self.seg_decoder(feat_act)  # upsample x32 total -> back to input res
        if seg_logits.shape[-2:] != x.shape[-2:]:
            seg_logits = F.interpolate(seg_logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return cls_logits, seg_logits, feat_act  # return feat_act too, for Grad-CAM hooking later


model = MultiTaskCXRModel(num_classes=len(CHEXPERT_CLASSES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. {n_params/1e6:.1f}M parameters.")

## 5. Losses, optimizer, checkpointing utilities

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs_flat = probs.view(probs.size(0), -1)
    targets_flat = targets.view(targets.size(0), -1)
    intersection = (probs_flat * targets_flat).sum(1)
    union = probs_flat.sum(1) + targets_flat.sum(1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


def seg_loss_fn(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dsc = dice_loss(logits, targets)
    return bce + dsc


cls_loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)


def save_checkpoint(model, optimizer, epoch, metrics, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "classes": CHEXPERT_CLASSES,
        "img_size": CONFIG["IMG_SIZE"],
    }, path)


def load_checkpoint(path, model, optimizer=None, device=DEVICE):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt

## 6. Build DataLoaders
Fill these in once your manifests exist. This cell is written defensively — it
will skip datasets that aren't found rather than crash, so you can smoke-test the
training loop with whatever's available.

In [ ]:
from torch.utils.data import ConcatDataset

train_datasets, val_datasets = [], []

if Path(CONFIG["CHEXPERT_TRAIN_CSV"]).exists():
    train_datasets.append(CheXpertDataset(CONFIG["CHEXPERT_TRAIN_CSV"], CONFIG["CHEXPERT_ROOT"], CHEXPERT_CLASSES, train_tf))
if Path(CONFIG["CHEXPERT_VALID_CSV"]).exists():
    val_datasets.append(CheXpertDataset(CONFIG["CHEXPERT_VALID_CSV"], CONFIG["CHEXPERT_ROOT"], CHEXPERT_CLASSES, eval_tf))

siim_manifest = Path(CONFIG["SIIM_ROOT"]) / "manifest.csv"
if siim_manifest.exists():
    full_seg_ds = PneumothoraxSegDataset(siim_manifest, CONFIG["SIIM_ROOT"], CONFIG["SIIM_ROOT"], CHEXPERT_CLASSES, train_tf, IMG_SIZE)
    n_val = max(1, int(0.1 * len(full_seg_ds)))
    n_train = len(full_seg_ds) - n_val
    seg_train, seg_val = torch.utils.data.random_split(full_seg_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))
    train_datasets.append(seg_train)
    val_datasets.append(seg_val)

if not train_datasets:
    print("!! No datasets found yet. Populate CONFIG paths / manifests, then re-run this cell.")
    train_loader = val_loader = None
else:
    train_ds = ConcatDataset(train_datasets)
    val_ds = ConcatDataset(val_datasets) if val_datasets else None
    train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True,
                               num_workers=CONFIG["NUM_WORKERS"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                             num_workers=CONFIG["NUM_WORKERS"], pin_memory=True) if val_ds else None
    print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds) if val_ds else 0}")

## 7. Training loop (per-epoch checkpointing + early stopping)

In [ ]:
def run_epoch(model, loader, optimizer=None, seg_weight=1.0, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_cls_loss, total_seg_loss, n_batches = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(is_train):
        pbar = tqdm(loader, desc="train" if is_train else "val", leave=False)
        for batch in pbar:
            images = batch["image"].to(device)
            labels = batch["labels"].to(device)
            masks = batch["mask"].to(device)
            has_mask = batch["has_mask"].to(device)  # [B]

            cls_logits, seg_logits, _ = model(images)
            cls_loss = cls_loss_fn(cls_logits, labels)

            if has_mask.sum() > 0:
                idx = has_mask.bool()
                seg_loss = seg_loss_fn(seg_logits[idx], masks[idx])
            else:
                seg_loss = torch.tensor(0.0, device=device)

            loss = cls_loss + seg_weight * seg_loss

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_cls_loss += cls_loss.item()
            total_seg_loss += seg_loss.item() if torch.is_tensor(seg_loss) else seg_loss
            n_batches += 1
            pbar.set_postfix(loss=total_loss / n_batches)

    return {
        "loss": total_loss / max(n_batches, 1),
        "cls_loss": total_cls_loss / max(n_batches, 1),
        "seg_loss": total_seg_loss / max(n_batches, 1),
    }


def train_model(model, train_loader, val_loader, epochs, seg_weight, log_path, ckpt_dir, patience):
    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_metrics = run_epoch(model, train_loader, optimizer, seg_weight)
        val_metrics = run_epoch(model, val_loader, None, seg_weight) if val_loader else {"loss": train_metrics["loss"], "cls_loss": 0, "seg_loss": 0}
        scheduler.step(val_metrics["loss"])
        dt = time.time() - t0

        row = {"epoch": epoch, "time_sec": round(dt, 1),
               "train_loss": train_metrics["loss"], "train_cls_loss": train_metrics["cls_loss"], "train_seg_loss": train_metrics["seg_loss"],
               "val_loss": val_metrics["loss"], "val_cls_loss": val_metrics["cls_loss"], "val_seg_loss": val_metrics["seg_loss"],
               "lr": optimizer.param_groups[0]["lr"]}
        history.append(row)
        print(f"Epoch {epoch}/{epochs} | train_loss={row['train_loss']:.4f} val_loss={row['val_loss']:.4f} ({dt:.0f}s)")

        pd.DataFrame(history).to_csv(log_path, index=False)
        save_checkpoint(model, optimizer, epoch, row, f"{ckpt_dir}/epoch_{epoch:02d}.pt")

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            epochs_no_improve = 0
            save_checkpoint(model, optimizer, epoch, row, f"{ckpt_dir}/final_model.pt")
            print(f"  -> new best model saved (val_loss={best_val_loss:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    return history

In [ ]:
# ---- Run training (only executes if a train_loader was built above) ----
if train_loader is not None:
    history = train_model(
        model, train_loader, val_loader,
        epochs=CONFIG["PHASE_A_EPOCHS"] + CONFIG["PHASE_B_EPOCHS"],
        seg_weight=CONFIG["SEG_LOSS_WEIGHT"],
        log_path=f"{CONFIG['LOG_DIR']}/training_log.csv",
        ckpt_dir=CONFIG["CHECKPOINT_DIR"],
        patience=CONFIG["EARLY_STOP_PATIENCE"],
    )
else:
    print("Skipping training — no data loaded. Populate dataset paths in CONFIG and re-run from Section 6.")

In [ ]:
# ---- Plot training curves ----
log_path = f"{CONFIG['LOG_DIR']}/training_log.csv"
if Path(log_path).exists():
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col in zip(axes, ["train_loss", "train_cls_loss", "train_seg_loss"]):
        ax.plot(df["epoch"], df[col], label=col)
        val_col = col.replace("train", "val")
        if val_col in df.columns:
            ax.plot(df["epoch"], df[val_col], label=val_col)
        ax.set_xlabel("epoch"); ax.legend(); ax.set_title(col)
    plt.tight_layout(); plt.show()
else:
    print("No training log yet.")

## 8. Smoke test on `final_model.pt`

Loads the checkpoint **fresh** (not the in-memory trained model) and verifies:
- correct output shapes / no NaNs
- segmentation head produces a non-trivial mask on a positive case
- Grad-CAM centroid falls inside the lung / mask region, and is NOT symmetric across
  both lungs on a known-unilateral case

In [ ]:
def compute_gradcam(model, image_tensor, target_class_idx, device=DEVICE):
    """Grad-CAM using the model's final backbone feature map (post-ReLU)."""
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device).requires_grad_(False)

    activations = {}
    def fwd_hook(module, inp, out):
        activations["value"] = out
    def bwd_hook(module, grad_in, grad_out):
        activations["grad"] = grad_out[0]

    target_layer = model.features.denseblock4.denselayer16.conv2  # last conv in DenseNet-121
    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    cls_logits, seg_logits, _ = model(image_tensor)
    score = cls_logits[0, target_class_idx]
    model.zero_grad()
    score.backward()

    h1.remove(); h2.remove()

    acts = activations["value"][0]      # [C, h, w]
    grads = activations["grad"][0]      # [C, h, w]
    weights = grads.mean(dim=(1, 2))    # [C]
    cam = torch.relu((weights[:, None, None] * acts).sum(0))
    cam = cam / (cam.max() + 1e-8)
    cam = cam.detach().cpu().numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    return cam, torch.sigmoid(score).item(), torch.sigmoid(seg_logits[0, 0]).detach().cpu().numpy()


def cam_centroid(cam, threshold=0.5):
    ys, xs = np.where(cam > threshold)
    if len(xs) == 0:
        return None
    return xs.mean(), ys.mean()


def run_smoke_test(checkpoint_path, test_images, device=DEVICE):
    """test_images: list of dicts like
    {"path": "...", "expected": "positive"/"negative", "side": "left"/"right"/None}"""
    fresh_model = MultiTaskCXRModel(num_classes=len(CHEXPERT_CLASSES)).to(device)
    ckpt = load_checkpoint(checkpoint_path, fresh_model)
    fresh_model.eval()

    report = {"checkpoint": checkpoint_path, "epoch": ckpt.get("epoch"), "checks": []}

    for item in test_images:
        result = {"image": item["path"], "expected": item["expected"], "pass": True, "notes": []}
        if not Path(item["path"]).exists():
            result["pass"] = False
            result["notes"].append("file not found")
            report["checks"].append(result)
            continue

        img = Image.open(item["path"]).convert("RGB")
        img_t = eval_tf(img)

        cam, ptx_prob, seg_mask = compute_gradcam(fresh_model, img_t, PNEUMOTHORAX_IDX, device)
        result["pneumothorax_prob"] = round(ptx_prob, 4)

        if np.isnan(cam).any() or np.isnan(ptx_prob):
            result["pass"] = False; result["notes"].append("NaN in output")

        left_mass = cam[:, :IMG_SIZE // 2].sum()
        right_mass = cam[:, IMG_SIZE // 2:].sum()
        total_mass = left_mass + right_mass + 1e-8
        result["left_frac"] = round(float(left_mass / total_mass), 3)
        result["right_frac"] = round(float(right_mass / total_mass), 3)

        if item["expected"] == "positive":
            if ptx_prob < 0.5:
                result["notes"].append("WARNING: expected positive but model predicted low probability")
            if item.get("side") in ("left", "right"):
                dominant = "left" if left_mass > right_mass else "right"
                if dominant != item["side"]:
                    result["pass"] = False
                    result["notes"].append(f"FAIL laterality: expected {item['side']}-dominant, got {dominant}-dominant")
                elif abs(result["left_frac"] - result["right_frac"]) < 0.15:
                    result["notes"].append("WARNING: heatmap is near-symmetric on a known unilateral case")
            if seg_mask.max() < 0.3:
                result["notes"].append("WARNING: segmentation head produced a near-empty mask on a positive case")
        elif item["expected"] == "negative":
            if ptx_prob > 0.5:
                result["notes"].append("WARNING: expected negative but model predicted high probability")

        report["checks"].append(result)

    report["all_pass"] = all(c["pass"] for c in report["checks"])
    return report


# ---- Fill in with real held-out images before trusting results ----
test_images = [
    {"path": "/data/holdout/known_positive_left.png", "expected": "positive", "side": "left"},
    {"path": "/data/holdout/known_negative.png", "expected": "negative", "side": None},
]

final_ckpt = f"{CONFIG['CHECKPOINT_DIR']}/final_model.pt"
if Path(final_ckpt).exists():
    report = run_smoke_test(final_ckpt, test_images)
    with open("smoke_test_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report, indent=2))
    print("\nSMOKE TEST:", "PASS" if report["all_pass"] else "FAIL — see notes above before shipping this checkpoint")
else:
    print(f"No checkpoint found at {final_ckpt} yet — run training first.")

## 9. Visual check — original vs Grad-CAM side by side

In [ ]:
def visualize_gradcam(checkpoint_path, image_path, device=DEVICE):
    fresh_model = MultiTaskCXRModel(num_classes=len(CHEXPERT_CLASSES)).to(device)
    load_checkpoint(checkpoint_path, fresh_model)
    fresh_model.eval()

    img = Image.open(image_path).convert("RGB")
    img_t = eval_tf(img)
    cam, prob, seg_mask = compute_gradcam(fresh_model, img_t, PNEUMOTHORAX_IDX, device)

    img_resized = np.array(img.resize((IMG_SIZE, IMG_SIZE)))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_resized, 0.5, heatmap, 0.5, 0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_resized); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title(f"Grad-CAM (P={prob:.2f})"); axes[1].axis("off")
    axes[2].imshow(seg_mask, cmap="gray"); axes[2].set_title("Segmentation head output"); axes[2].axis("off")
    plt.tight_layout(); plt.show()

# Example usage once you have a checkpoint + test image:
# visualize_gradcam(f"{CONFIG['CHECKPOINT_DIR']}/final_model.pt", "/data/holdout/known_positive_left.png")